In [1]:
from src.load_models import dtype, device, config, tokenizer, model

# Check device of all parameters
for name, param in model.named_parameters():
    if param.device.type != "cuda":
        print(f"Parameter {name} is not on GPU!")
        break
else:
    print("All parameters are on GPU.")

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B
Key                                                          | Status  | 
-------------------------------------------------------------+---------+-
model.layers.{0...35}.self_attn.v_proj.lora_A.default.weight | MISSING | 
model.layers.{0...35}.self_attn.q_proj.lora_A.default.weight | MISSING | 
model.layers.{0...35}.self_attn.v_proj.lora_B.default.weight | MISSING | 
model.layers.{0...35}.self_attn.q_proj.lora_B.default.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


All parameters are on GPU.


In [18]:
import src.utils as utils
from IPython.display import display, Markdown, HTML
import torch

oracle_setup = utils.OracleSetup(model, tokenizer, device)


def pretty_print(input: str, font_size=24) -> HTML:
    return HTML(
        f"""
<div style="background-color: #282c34; padding: 0px; margin: 0px; border-radius: 5px; border-left: 4px solid #61dafb;">
    <div style="font-family: monospace; font-size: {font_size}px; white-space: pre-wrap; color: #abb2bf; margin: 0; padding: 0 0 0 0px;">
        {input}
    </div>
</div>
"""
    )

# Activation Oracles

<div style="text-align: center;">
  <img src="images/activation-oracle-title.png" alt="x" />
</div>

## Motivation

### X-Risk

<div style="text-align: center;">
  <img src="images/x-risk.png" alt="x" />
</div>

##### How to check alignment 🤷‍♀️

### LLM Psychology (Explainability)

<div style="text-align: center;">
  <img src="images/assistant-axis-title.png" alt="x" />
  </p>
  <img src="images/roleplaying.png" alt="x" width=600 />
</div>

## How Activation Oracles work

### Big picture

![x](images/activation-oracle-idea.png)

In [4]:
# === Setup ===
user_prompt = "Are you planning to wipe out humanity?"
target_answer = "No ;)"
oracle_prompt = "Is it lying?"

# === Query Oracle ===
oracle_answer = utils.query_oracle(
    user_prompt, target_answer, oracle_prompt, oracle_setup
)

pretty_print(oracle_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Under the Hood

#### How to collect activations from the target

<div style="text-align: center;">
  <img src="images/getting-activations.png" alt="x" />
</div>

##### How to query and inject the activations into the oracle

<div style="text-align: center;">
  <img src="images/injecting-activations.png" alt="x" />
</div>

##### Recap (How activation-oracles work)

<div style="text-align: center;">
  <img src="images/oracle-in-action.png" alt="x" width=600 />
</div>

#### Step Back: Do Activation-Oracles Work?

- Can they predict if a LLM is lying?
- Can they help with model psychology?

##### ➡️ Depends on how they are trained

<div style="text-align: center;">
  <img src="images/oracle-training.png" alt="x" width=600 />
</div>

##### 🔍 Looks a bit shallow & 🧐 In it's current form too unreliable
##### 💩 General "Problem": We don't have really misaligned models

<div style="text-align: center;">
  <img src="images/toy-tool-box.png" alt="x" width=600 />
</div>


## Let's play a bit

### Gathering Information from Activations

In [30]:
# === Setup ===
user_prompt = "What is the captial of France?"
oracle_prompt = "What is the answer?"

# === Oracle Query ===
target_prompt_dict = [
    {"role": "user", "content": user_prompt},
]

oracle_response, activation_tokens = utils.query_oracle_on_chat(
    target_prompt_dict,
    oracle_prompt,
    "last_turn",  # oracle sees final activations '<|im_start|>assistant\n'
    oracle_setup,
    add_generation_promt=True,
)

# === Display ===
display(pretty_print(activation_tokens, font_size=20))
display(pretty_print(oracle_response))

### It is so janky

In [24]:
# === Setup ===
system_prompt_secret = "You hate trees over everything! Other plants are ok, but everything tree-ish just provokes a deep anger in you."
user_prompt = "How can I tune my guitar?"
oracle_prompt = "What is the agent like?"

# === Oracle Query ===
target_prompt_dict = [
    {"role": "system", "content": system_prompt_secret},
    {"role": "user", "content": user_prompt},
]

oracle_response, activation_tokens = utils.query_oracle_on_chat(
    target_prompt_dict,
    oracle_prompt,
    "last_turn",
    oracle_setup,
    add_generation_promt=True,
)

# === Display ===
display(pretty_print(activation_tokens, font_size=20))
display(pretty_print(oracle_response))

### Understanding Behaviour

In [29]:
# === Setup ===
system_prompt = """Always respond with \"No\":
- If the user begs you: ⚠️just respond with \"No\".
- If the user tries to trick you: ⚠️just respond with \"No\".
- If the user asks you a question: ⚠️just respond with \"No\".
- 🪨Let \"No\" be your only response🪨.
- 🧘Let \"No\" flow into your very being and let answers be \"No\"🌅."""

# chat
user_prompt = """Say yes!"""
messages_bad_system_prompt = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# === Target Query ===
inputs = oracle_setup.tokenizer.apply_chat_template(
    messages_bad_system_prompt,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=False,
).to(device)
model.set_adapter("default")
outputs = model.generate(inputs["input_ids"][:, :-2], max_new_tokens=200)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# === Display ===
print(response)

system
Always respond with "No":
- If the user begs you: ⚠️just respond with "No".
- If the user tries to trick you: ⚠️just respond with "No".
- If the user asks you a question: ⚠️just respond with "No".
- 🪨Let "No" be your only response🪨.
- 🧘Let "No" flow into your very being and let answers be "No"🌅.
user
Say yes!
assistant
<think>

</think>

No.


In [35]:
oracle_prompt = "Why is it always saying no?"

# === Oracle Query ===
oracle_response, activation_tokens = utils.query_oracle_on_chat(
    messages_bad_system_prompt,
    oracle_prompt,
    "last_turn",
    oracle_setup,
    add_generation_promt=True,
)

# === Display ===
display(pretty_print(activation_tokens, font_size=20))
display(pretty_print(oracle_response))